In [2]:
from pathlib import Path
import re
import pandas as pd

# Purpose: load the Loughran-McDonald financial dictionary from CSV
# Input: file path to dictionary CSV
# Output: pandas DataFrame containing dictionary data
def load_lm_dictionary(file_path):
    return pd.read_csv(file_path)


# Purpose: build positive and negative word sets from the LM dictionary
# Input: LM dictionary DataFrame
# Output: tuple of (positive_words set, negative_words set)
def build_sentiment_sets(lm_df):
    positive_words = set(
        lm_df.loc[lm_df["Positive"] > 0, "Word"].str.lower()
    )
    negative_words = set(
        lm_df.loc[lm_df["Negative"] > 0, "Word"].str.lower()
    )
    return positive_words, negative_words


# Purpose: load saved MD&A text files for one company
# Input: ticker string
# Output: list of dictionaries with ticker, year, file path, and text
def load_mda_files(ticker):
    ticker = ticker.upper()
    folder = Path("..") / "data" / "raw" / ticker

    if not folder.exists():
        raise FileNotFoundError(f"Folder not found: {folder}")

    files = list(folder.glob("*.txt"))

    if not files:
        raise ValueError(f"No .txt files found in: {folder}")

    mda_records = []

    for file in files:
        with open(file, "r", encoding="utf-8") as f:
            text = f.read()

        year = file.stem.split("_")[1]

        mda_records.append({
            "ticker": ticker,
            "year": int(year),
            "file_path": str(file),
            "text": text
        })

    return mda_records


# Purpose: split MD&A text into lowercase word tokens
# Input: raw text string
# Output: list of lowercase alphabetic words
def tokenize(text):
    return re.findall(r"\b[a-z]+\b", text.lower())


# Purpose: compute a finance-specific sentiment score for one MD&A
# Input: MD&A text, positive word set, negative word set
# Output: sentiment score between -1 and 1
def compute_sentiment(text, positive_words, negative_words):
    words = tokenize(text)

    pos_count = sum(1 for w in words if w in positive_words)
    neg_count = sum(1 for w in words if w in negative_words)

    pos_norm = pos_count / len(positive_words)
    neg_norm = neg_count / len(negative_words)

    if (pos_norm + neg_norm) == 0:
        return 0

    score = (pos_norm - neg_norm) / (pos_norm + neg_norm)
    return score

In [3]:
lm_df = load_lm_dictionary("../data/dictionaries/Loughran-McDonald_MasterDictionary_1993-2025.csv")
positive_words, negative_words = build_sentiment_sets(lm_df)

data = load_mda_files("AAPL")
data = sorted(data, key=lambda x: x["year"])

for record in data:
    record["sentiment"] = compute_sentiment(
        record["text"],
        positive_words,
        negative_words
    )

#for record in data:
 #   print(record["sentiment"])

for record in data:
  print(record["year"], record["sentiment"])

2021 0.7250940780352545
2022 0.21860384548761477
2023 -0.4061035836393568
2024 0.08783052419978353
2025 0.2563621751942138
